## Importovanie knižníc

In [ ]:
import pandas as pd
import matplotlib
import seaborn as sns

### Analýza štruktúr dát ako súbory
Dataset sa skladá z 3 csv súborov `units.csv`, `readings.csv`, `gateways.csv`

### Readings.csv

Súbor používa znak tabulátora ako delimiter. Stĺpec *ts_cycle* obsahoval zmiešané formáty dátumov a časov, takže sme ich zjednotili pomocou parametra `parse_dates` vo funkcii `read_csv`. Okrem toho sme chceli zjednotiť ID vozidiel *id_unit*, tak sme ho prekonvertovali na `int` po odstránení "U" zo začiatku stringu.

In [ ]:
READINGS = "readings.csv"
readings_df = pd.read_csv(READINGS, delimiter="\t", parse_dates=["ts_cycle"], date_format="mixed")

readings_df["id_unit"] = readings_df["id_unit"].str.replace("U", "").astype(int)

readings_df

,id_unit,ts_cycle,motor_current,dpFilter,temperature_oil,reservoirPressure,oilPressure,rms_vibration,Flowmeter,humidityIntake,failure
0,1713,2023-04-12 00:00:00,4.53942,17.07120,41.44256,8.31174,3.50095,5.54202,26.60877,43.95098,0
1,252,2023-01-20 00:00:00,2.32227,33.57520,67.51767,7.95208,2.84785,2.65594,28.66642,31.12998,0
2,822,2023-08-30 00:00:00,7.47554,33.60901,66.09002,9.86046,4.40705,NaN,29.12791,87.72845,0
3,1266,2023-03-11 00:00:00,7.58163,42.41991,54.71008,7.80329,2.91898,2.47698,36.47254,37.02388,0
4,1034,2023-06-09 00:00:00,7.34114,17.23080,46.08248,9.06460,4.06547,2.60575,28.06168,65.65696,0
5,1618,2023-06-23 00:00:00,2.93551,15.57236,57.85095,8.14446,2.99617,1.16123,22.18940,33.39083,0
6,1439,2023-04-12 00:00:00,5.73117,22.55021,47.34223,9.40355,4.51815,1.80400,30.14390,23.39231,0
7,399,2023-11-24 00:00:00,3.09522,15.22283,42.07898,8.75411,3.73779,4.93936,30.31024,64.08892,0
8,1074,2023-10-11 00:00:00,7.32504,16.90241,49.08737,9.16743,4.62081,2.20880,24.82717,NaN,1
9,1364,2023-09-21 19:47:00,7.08560,19.44801,61.05277,8.36557,3.33240,4.64916,29.16119,12.49991,0


### Units.csv

Súbor používa znak "," ako delimiter. Stĺpec *dutyClass* obsahuje rôzne pomenovania pre pravdepodobne rovnaké kategórie, no zatiaľ nevieme ktoré môžme zlúčiť.

In [49]:
UNITS = "units.csv"
units_df = pd.read_csv(UNITS, delimiter=",")

units_df

,id_unit,serial_number,year_commission,dutyClass,hours_service,operatorOwner,idDepot
0,1,UJ-6276-85840,2021,N,10733.0,Teixeira Macedo e Filhos,50
1,2,CE-5946-76267,2017,2,26763.3,Benthin Huhn GmbH & Co. KGaA,59
2,3,JH-8720-55338,2023,2,NaN,"Brown, Glover and Williams",68
3,4,OT-1786-88207,2022,2,NaN,Bolnbach,70
4,5,GA-2070-91868,2023,heavy,6964.2,Trapp GmbH,76
...,...,...,...,...,...,...,...
1995,1996,AZ-3337-16607,2019,L,19159.1,Crawford-Mccarthy,44
1996,1997,LY-8013-32401,2024,light,3156.0,Meister,11
1997,1998,BE-8220-94317,2023,N,4647.1,Matias,13
1998,1999,GT-7617-20587,2023,3,5990.0,Spears and Sons,113


### Gateways.csv

Tento súbor používa znak ";" ako delimiter. Stĺpce *last_seen* a *dateInstall* oba používajú nejednotné formáty času, tak sme sa ich rozhodli zjednotiť.
Z povahy dát usudzujeme, že stĺpec *id_unit* bude značiť ID vozidla v databáze, do ktorého bola daný smerovač osadený. Ak smerovač ID vozidla nemá, znamená to že ho nevieme k žiadnemu priradiť a je pre nás irelevantný -> `dropna`. Stĺpec *id_unit* bol uložený ako typ `float`, tak sme ho prekonvertovali na typ `int`.

In [48]:
GATEWAYS = "gateways.csv"
gateways_df = pd.read_csv(GATEWAYS, delimiter=";", parse_dates=["last_seen", "dateInstall"], date_format="mixed")

gateways_df = gateways_df.dropna(subset="id_unit")
gateways_df["id_unit"] = gateways_df.id_unit.astype(int)

gateways_df


,gatewayId,lat,model_hardware,version_firmware,Technician,mac_address,idDepot,id_unit,vendor,Lon,Ipv4,last_seen,dbm_signal,dateInstall
0,GW-00000,40.92527,NODE-7,v1.2.5,Mrs. Stephanie Bartlett,74:f3:ef:c4:30:c9,107,449,Preiß GmbH & Co. KG,-9.21042,200.82.55.76,2024-02-11,-76.6,2015-06-20
1,GW-00001,41.03087,NODE-9,v2.7.0,Hans-Otto Weitzel,a2:3f:d6:07:bb:cb,61,1499,Ditschlerin Bruder AG,-8.35210,75.246.136.39,2024-09-03,-73.4,2016-12-19
2,GW-00002,40.08425,NODE-9,v1.2.5,John Camacho,b4:4b:8e:a1:fc:2c,87,1363,Caspar GmbH & Co. KGaA,-7.23610,127.148.71.104,2025-01-15,-67.9,2022-09-17
3,GW-00003,38.91664,NODE-9,v2.7.0,Elisa Noack-Rogner,fb:c3:5d:4e:44:27,64,447,Preiß GmbH & Co. KG,-8.58157,109.188.241.62,2024-06-09,-56.7,2021-12-17
4,GW-00004,41.54055,NODE-7,v3.4.1,Shelby Wells,80:d3:e7:6f:07:b8,6,308,Tintzmann Hoffmann AG,-8.61346,93.62.149.214,2024-02-27,-58.1,2017-06-27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,GW-01995,41.45447,GWX-100,v1.2.5,Austin Cole,ab:2b:14:c4:18:ad,73,495,Assunção Neto e Filhos,-7.99980,208.190.223.152,2024-05-26,-81.2,2023-10-20
1996,GW-01996,39.68046,GWX-100,v2.7.0,Tammy Miller,c3:88:22:ca:dc:9f,3,1182,Vasquez LLC,-6.92858,94.155.217.238,2024-08-05,-81.4,2017-03-21
1997,GW-01997,40.55435,GWX-200,v2.7.0,Mona Harloff-Ebert,0e:ca:54:69:15:dd,51,1044,Vasquez LLC,-8.50850,192.14.20.128,2024-09-05,-72.9,2023-06-24
1998,GW-01998,40.57805,NODE-7,v3.4.1,Joseph Smith,af:7e:e8:29:17:38,28,1256,Caspar GmbH & Co. KGaA,-6.89126,168.254.175.77,2024-02-14,-93.7,2019-12-10


,id_unit,serial_number,year_commission,dutyClass,hours_service,operatorOwner,idDepot
0,1,UJ-6276-85840,2021,N,10733.0,Teixeira Macedo e Filhos,50
1,2,CE-5946-76267,2017,2,26763.3,Benthin Huhn GmbH & Co. KGaA,59
2,3,JH-8720-55338,2023,2,NaN,"Brown, Glover and Williams",68
3,4,OT-1786-88207,2022,2,NaN,Bolnbach,70
4,5,GA-2070-91868,2023,heavy,6964.2,Trapp GmbH,76
...,...,...,...,...,...,...,...
1995,1996,AZ-3337-16607,2019,L,19159.1,Crawford-Mccarthy,44
1996,1997,LY-8013-32401,2024,light,3156.0,Meister,11
1997,1998,BE-8220-94317,2023,N,4647.1,Matias,13
1998,1999,GT-7617-20587,2023,3,5990.0,Spears and Sons,113
